In [1]:
import warnings
warnings.filterwarnings("ignore")

from datetime import datetime
import pandas as pd
from pyspark.sql import SparkSession
import pyspark.sql.functions as f
from pyspark.sql import Window
from IPython.display import display

pd.DataFrame.iteritems = pd.DataFrame.items

In [2]:
spark = SparkSession.builder\
       .master("local[*]")\
       .appName("VariableSelectionML")\
       .config("spark.executor.memory", "8g")\
       .config("spark.driver.memory", "6g")\
       .config("spark.driver.extraJavaOptions",
                "--add-opens java.base/sun.net.www.protocol.http=ALL-UNNAMED "
                "--add-opens java.base/sun.net.www.protocol.https=ALL-UNNAMED "
                "--add-opens java.base/sun.net.www.protocol.jar=ALL-UNNAMED")\
       .config("spark.executor.extraJavaOptions",
                "--add-opens java.base/sun.net.www.protocol.http=ALL-UNNAMED "
                "--add-opens java.base/sun.net.www.protocol.https=ALL-UNNAMED "
                "--add-opens java.base/sun.net.www.protocol.jar=ALL-UNNAMED")\
       .getOrCreate()

In [3]:
spark.sparkContext

<SparkContext master=local[*] appName=VariableSelectionML>

In [4]:
# pip install h2o_pysparkling_3.5

In [5]:
from pysparkling import *
import h2o
conf = H2OConf().setLogLevel("ERROR") # WARN
hc = H2OContext.getOrCreate(conf)

Connecting to H2O server at http://6b0fa5dc04cb:54323 ... successful.
Please download and install the latest version from: https://h2o-release.s3.amazonaws.com/h2o/latest_stable.html


H2O_cluster_uptime:,05 secs
H2O_cluster_timezone:,Etc/UTC
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.6
H2O_cluster_version_age:,"1 year, 4 months and 21 days"
H2O_cluster_name:,sparkling-water-jovyan_local-1774286106606
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,5.887 Gb
H2O_cluster_total_cores:,12
H2O_cluster_allowed_cores:,12
H2O_cluster_status:,"locked, healthy"



Sparkling Water Context:
 * Sparkling Water Version: 3.46.0.6-1-3.5
 * H2O name: sparkling-water-jovyan_local-1774286106606
 * cluster size: 1
 * list of used nodes:
  (executorId, host, port)
  ------------------------
  (0,172.17.0.2,54321)
  ------------------------

  Open H2O Flow in browser: http://6b0fa5dc04cb:54323 (CMD + click in Mac OSX)

    


In [6]:
frame = h2o.import_file("loan.csv")
df = hc.asSparkFrame(frame)

window = Window.orderBy(f.lit('A'))
df = df.select(f.row_number().over(window).alias("id"), "*")

display(df.limit(10).toPandas())

Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%


,id,loan_amnt,term,int_rate,emp_length,home_ownership,annual_inc,purpose,addr_state,dti,delinq_2yrs,revol_util,total_acc,bad_loan,longest_credit_length,verification_status
0,1,5000,36 months,10.65,10,RENT,24000.0,credit_card,AZ,27.65,0,83.7,9,0,26,verified
1,2,2500,60 months,15.27,0,RENT,30000.0,car,GA,1.00,0,9.4,4,1,12,verified
2,3,2400,36 months,15.96,10,RENT,12252.0,small_business,IL,8.72,0,98.5,10,0,10,not verified
3,4,10000,36 months,13.49,10,RENT,49200.0,other,CA,20.00,0,21.0,37,0,15,verified
4,5,5000,36 months,7.90,3,RENT,36000.0,wedding,AZ,11.20,0,28.3,12,0,7,verified
5,6,3000,36 months,18.64,9,RENT,48000.0,car,CA,5.35,0,87.5,4,0,4,verified
6,7,5600,60 months,21.28,4,OWN,40000.0,small_business,CA,5.55,0,32.6,13,1,7,verified
7,8,5375,60 months,12.69,0,RENT,15000.0,other,TX,18.08,0,36.5,3,1,7,verified
8,9,6500,60 months,14.65,5,OWN,72000.0,debt_consolidation,AZ,16.12,0,20.6,23,0,13,not verified
9,10,12000,36 months,12.69,10,OWN,75000.0,debt_consolidation,CA,10.78,0,67.1,34,0,22,verified


In [7]:
target_name_in_dataset = "bad_loan"

seed = 47  # set seed to your own number for reproducibility

In [8]:
col_list = [colu.lower() for colu in df.columns]

print(f"Columns: {len(col_list)}")

df = df.select(col_list)

print(f"Rows: {df.count()}")

agg_tab = df.agg(f.count(f.lit(1)).alias("count")
        , f.mean(target_name_in_dataset).alias(f"mean_{target_name_in_dataset}"))\
        .withColumn(f"mean_{target_name_in_dataset}", f.round(f"mean_{target_name_in_dataset}", 6))

display(agg_tab.toPandas())

Columns: 16
Rows: 999


,count,mean_bad_loan
0,999,0.194194


In [9]:
# coonstant or id columns to drop

cols_to_ignore =  ["id", "addr_state", target_name_in_dataset]

init_features_list = [colu for colu in col_list if colu.lower() not in cols_to_ignore]

print(f"Features for analysis: {len(init_features_list)}")

Features for analysis: 13


In [10]:
import pyspark.pandas as ps

approx_counts = df.agg(*(f.approx_count_distinct(f.col(c)).alias(c) for c in init_features_list))
psdf = approx_counts.pandas_api()
transposed_psdf = psdf.transpose()
transposed_psdf = transposed_psdf.reset_index()
transposed_psdf.columns = ['col_name', 'count_distinct']
approx_counts_transposed = transposed_psdf.to_spark()

counts = df.agg(*(f.count(f.col(c)).alias(c) for c in init_features_list))
psdf = counts.pandas_api()
transposed_psdf = psdf.transpose()
transposed_psdf = transposed_psdf.reset_index()
transposed_psdf.columns = ['col_name', 'count']
counts_transposed = transposed_psdf.to_spark()

approx_counts_transposed = approx_counts_transposed.join(counts_transposed, on='col_name')

n = df.count()
approx_counts_transposed = approx_counts_transposed.withColumn("total", f.lit(n))
approx_counts_transposed = approx_counts_transposed.withColumn("missing", f.col("total") - f.col("count"))
approx_counts_transposed = approx_counts_transposed.withColumn("pct_missing", f.col("missing")/f.col("total"))
approx_counts_transposed = approx_counts_transposed.withColumn("pct_missing", f.round("pct_missing", 4))
approx_counts_transposed = approx_counts_transposed.withColumn("pct_not_missing", f.col("count")/f.col("total"))
approx_counts_transposed = approx_counts_transposed.withColumn("pct_not_missing", f.round("pct_not_missing", 4))
approx_counts_transposed = approx_counts_transposed.select("col_name", "count_distinct", "missing", "pct_missing", "pct_not_missing", "total")

approx_counts_transposed.show(20, False)

+---------------------+--------------+-------+-----------+---------------+-----+
|col_name             |count_distinct|missing|pct_missing|pct_not_missing|total|
+---------------------+--------------+-------+-----------+---------------+-----+
|annual_inc           |281           |0      |0.0        |1.0            |999  |
|delinq_2yrs          |4             |0      |0.0        |1.0            |999  |
|dti                  |811           |0      |0.0        |1.0            |999  |
|emp_length           |11            |17     |0.017      |0.983          |999  |
|home_ownership       |3             |0      |0.0        |1.0            |999  |
|int_rate             |32            |0      |0.0        |1.0            |999  |
|loan_amnt            |220           |0      |0.0        |1.0            |999  |
|longest_credit_length|35            |0      |0.0        |1.0            |999  |
|purpose              |13            |0      |0.0        |1.0            |999  |
|revol_util           |596  

In [11]:
from pysparkling.ml import H2OXGBoostClassifier

approx_counts_transposed = approx_counts_transposed.withColumn("trai_gini", f.lit(None))
approx_counts_transposed = approx_counts_transposed.withColumn("cv_gini", f.lit(None))
approx_counts_transposed = approx_counts_transposed.withColumn("reason", f.lit(None))
app_counts = approx_counts_transposed.toPandas()

print("--- {} Start".format(datetime.now().strftime("%Y/%m/%d %H:%M:%S")))

for i in range(len(app_counts)):
    feature = app_counts.iloc[i, app_counts.columns.get_loc('col_name')]
    n = app_counts.iloc[i, app_counts.columns.get_loc('count_distinct')]
    if n >= 2:
        try:
            estimator = H2OXGBoostClassifier(
                featuresCols = [feature],
                labelCol = target_name_in_dataset,
                nfolds = 5,
                evalMetric = 'aucpr', 
                parallelizeCrossValidation = True,
                seed = seed
            )
            model = estimator.fit(df.select(feature, target_name_in_dataset))
            trai_gini = model.getTrainingMetrics()['Gini']
            cv_gini = model.getCrossValidationMetrics()['Gini']
            app_counts.iloc[i, app_counts.columns.get_loc('reason')] = "ok"
            app_counts.iloc[i, app_counts.columns.get_loc('trai_gini')] = round(trai_gini, 6)
            app_counts.iloc[i, app_counts.columns.get_loc('cv_gini')] = round(cv_gini, 6)
        except Exception as e:
            app_counts.iloc[i, app_counts.columns.get_loc('reason')] = f"issue with feature {feature}: {e}"
            app_counts.iloc[i, app_counts.columns.get_loc('trai_gini')] = 0.0
            app_counts.iloc[i, app_counts.columns.get_loc('cv_gini')] = 0.0
    else:
        app_counts.iloc[i, app_counts.columns.get_loc('reason')] = "const value"
        app_counts.iloc[i, app_counts.columns.get_loc('trai_gini')] = 0.0
        app_counts.iloc[i, app_counts.columns.get_loc('cv_gini')] = 0.0

print("--- {} End".format(datetime.now().strftime("%Y/%m/%d %H:%M:%S")))        
 
approx_counts_transposed = spark.createDataFrame(app_counts)
approx_counts_transposed.orderBy(f.col('trai_gini').desc()).show(20, False)

--- 2026/03/23 17:15:20 Start
--- 2026/03/23 17:15:53 End
+---------------------+--------------+-------+-----------+---------------+-----+---------+---------+------+
|col_name             |count_distinct|missing|pct_missing|pct_not_missing|total|trai_gini|cv_gini  |reason|
+---------------------+--------------+-------+-----------+---------------+-----+---------+---------+------+
|dti                  |811           |0      |0.0        |1.0            |999  |0.759736 |0.04166  |ok    |
|revol_util           |596           |0      |0.0        |1.0            |999  |0.709541 |0.002324 |ok    |
|annual_inc           |281           |0      |0.0        |1.0            |999  |0.515093 |0.031811 |ok    |
|loan_amnt            |220           |0      |0.0        |1.0            |999  |0.454223 |-0.103477|ok    |
|int_rate             |32            |0      |0.0        |1.0            |999  |0.394429 |0.251822 |ok    |
|total_acc            |55            |0      |0.0        |1.0            |999 